In [ ]:
# Databricks notebook sourceMAGIC %md# BRONZE RAW POLICIES DATA BEFORE PREPROCESSING IN SILVER

In [ ]:
MAGIC %sqlSELECT * FROM bupa_bronze.policies

%md## Cell 1 — Import utilities & set environment

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# ==============================================# _01_policies_silver.ipynb# Enterprise Silver Layer for Policies# ==============================================# Import required PySpark modulesfrom pyspark.sql import functions as Ffrom pyspark.sql.types import *from datetime import datetime# Import utility functions from the utils notebook# Make sure you've run `_00_utils_silver` once in the same cluster# so its functions are registered in memory# If you modularized them as a Python file, use: from utils_silver import *# Otherwise just `%run ../_00_utils_silver`# Databases (Hive metastore)DB_SILVER = "bupa_silver"DB_REF    = "bupa_reference"spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_REF}")print("DB_SILVER:", DB_SILVER)print("DB_REF   :", DB_REF)

%md## Cell 2 — Read Bronze data & select/cast

In [ ]:
# ============ Load Bronze Policies ============BRONZE_POLICIES_PATH = "abfss://bronze@bupaprocesseddatastorage.dfs.core.windows.net/policies"try:    pol_bz = spark.read.delta(BRONZE_POLICIES_PATH)except:    pol_bz = spark.table("bupa_bronze.policies")pol = (pol_bz.select(        F.col("Policy_ID").cast("string"),        F.col("Customer_ID").cast("string"),        F.col("Product_Line").cast("string"),        F.col("Sum_Insured_GBP").cast("double"),        F.col("Annual_Premium_GBP").cast("double"),        F.to_date("Policy_Start_Date").alias("Policy_Start_Date"),        F.to_date("Policy_End_Date").alias("Policy_End_Date"),        F.col("Renewal_Offered_Flag").cast("int"),        F.col("Renewal_Accepted_Flag").cast("int"),        F.col("Discount_Offered_%").cast("double"),        F.col("Channel").cast("string")     ))print(f"Bronze rowcount: {pol.count()}")

%md## Cell 3 — Enforce schema & key checks

In [ ]:
# ============ Schema Enforcement & Key Checks ============policy_schema = StructType([    StructField("Policy_ID", StringType()),    StructField("Customer_ID", StringType()),    StructField("Product_Line", StringType()),    StructField("Sum_Insured_GBP", DoubleType()),    StructField("Annual_Premium_GBP", DoubleType()),    StructField("Policy_Start_Date", DateType()),    StructField("Policy_End_Date", DateType()),    StructField("Renewal_Offered_Flag", IntegerType()),    StructField("Renewal_Accepted_Flag", IntegerType()),    StructField("Discount_Offered_%", DoubleType()),    StructField("Channel", StringType())])pol = enforce_schema(pol, policy_schema)# Quarantine missing keyspol_bad = pol.filter(F.col("Policy_ID").isNull() | F.col("Customer_ID").isNull())pol_ok  = pol.filter(F.col("Policy_ID").isNotNull() & F.col("Customer_ID").isNotNull())if pol_bad.take(1):    quarantine(pol_bad, "null_key_policy_or_customer", "policies")print(f"After key check: {pol_ok.count()} rows")

%md## Cell 4 — Date fix + business rule validations

In [ ]:
# ---------- Business Validations (Soft vs Hard) ----------pol_ok = pol_ok.withColumn(    "dq_money_valid",    F.when((F.col("Annual_Premium_GBP") >= 0) & (F.col("Sum_Insured_GBP") >= 0), 1).otherwise(0)).withColumn(    "dq_discount_valid",    F.when(F.col("Discount_Offered_%").between(0, 100) | F.col("Discount_Offered_%").isNull(), 1).otherwise(0)).withColumn(    "dq_renewal_valid",    F.when(~((F.col("Renewal_Accepted_Flag") == 1) & (F.coalesce(F.col("Renewal_Offered_Flag"), F.lit(0)) != 1)), 1).otherwise(0))# Hard DQ: null PKs handled above, date order already fixed# Soft DQ: Keep flagged rows, downstream rules use SLA thresholds

%md## Cell 5 — Deduplicate + normalize + derived fields

In [ ]:
# ============ Deduplicate & Feature Engineering ============# Deduplicate (keep latest by Policy_Start_Date)pol_silver = drop_dupes_keep_latest(pol_ok, ["Policy_ID"], ["Policy_Start_Date"])# Normalize categoriespol_silver = (pol_silver    .withColumn("Product_Line", F.initcap(F.trim("Product_Line")))    .withColumn("Channel", F.initcap(F.trim("Channel"))))# Derived columnspol_silver = pol_silver.withColumn(    "Policy_Duration_Days",    F.when(F.col("Policy_End_Date").isNotNull() & F.col("Policy_Start_Date").isNotNull(),           F.datediff("Policy_End_Date", "Policy_Start_Date"))     .otherwise(F.lit(None)))pol_silver = pol_silver.withColumn(    "Renewal_Conversion",    F.when(F.col("Renewal_Offered_Flag") == 1, F.col("Renewal_Accepted_Flag")).otherwise(F.lit(None)))display(pol_silver.limit(10))

%md## Cell 6 — Reference dimensions setup

In [ ]:
# ============ Create Reference Tables ============spark.createDataFrame(    [("Agent",), ("Broker",), ("Online",), ("Partner",)],    "Channel STRING").write.mode("overwrite").saveAsTable(f"{DB_REF}.dim_channel")spark.createDataFrame(    [("Accident",), ("Dental",), ("Health",), ("Motor",), ("Travel",)],    "Product_Line STRING").write.mode("overwrite").saveAsTable(f"{DB_REF}.dim_product_line")print("dim_channel:")display(spark.table(f"{DB_REF}.dim_channel"))print("dim_product_line:")display(spark.table(f"{DB_REF}.dim_product_line"))

%md## Cell 7 — Data Quality (DQ) validations

In [ ]:
# ============ Data Quality (DQ) Rules) — Cell 7 (robust, SLA-gated renewal) ============from pyspark.sql import functions as F# --- Normalize renewal flags to clean 0/1 ints (avoid mixed types) ---def to_flag(colname):    s = F.lower(F.trim(F.col(colname).cast("string")))    return F.when(s.isin("1","y","yes","true","t","on"), 1) \            .when(s.isin("0","n","no","false","f","off"), 0) \            .otherwise(F.col(colname).cast("int"))  # if already 0/1, keeppol_silver = (pol_silver    .withColumn("Renewal_Offered_Flag",  to_flag("Renewal_Offered_Flag").cast("int"))    .withColumn("Renewal_Accepted_Flag", to_flag("Renewal_Accepted_Flag").cast("int")))# 1) Basic expectations (ERROR severity)dq_expect(pol_silver, "key_not_null",          "Policy_ID IS NOT NULL AND Customer_ID IS NOT NULL",          "error", "policies", quarantine, "null_key_policy_or_customer")dq_expect(pol_silver, "date_order",          "Policy_Start_Date IS NULL OR Policy_End_Date IS NULL OR Policy_End_Date >= Policy_Start_Date",          "error", "policies", quarantine, "date_order_violation")dq_expect(pol_silver, "money_non_negative",          "coalesce(Annual_Premium_GBP,0) >= 0 AND coalesce(Sum_Insured_GBP,0) >= 0",          "error", "policies", quarantine, "negative_money_values")dq_expect(pol_silver, "discount_bounds",          "`Discount_Offered_%` IS NULL OR (`Discount_Offered_%` BETWEEN 0 AND 100)",          "error", "policies", quarantine, "discount_out_of_bounds")# 2) Renewal consistency — SLA-gated (custom block)viol_df = pol_silver.filter(    (F.col("Renewal_Accepted_Flag") == 1) & (F.coalesce(F.col("Renewal_Offered_Flag"), F.lit(0)) != 1))viol_cnt = viol_df.count()total    = pol_silver.count()viol_pct = 0 if total == 0 else round(viol_cnt / total * 100.0, 4)if viol_cnt > 0:    quarantine(viol_df, "renewal_inconsistent", "policies")print(f"[RECON] renewal_inconsistent: {viol_cnt}/{total} ({viol_pct}%)")# Temporary SLA relaxed to unblock; tighten later (e.g., 0.5)SLA_PCT = 25.0if viol_pct > SLA_PCT:    raise Exception(f"DQ gate failed: policies · renewal_consistency {viol_pct}% > {SLA_PCT}%")else:    print("✅ DQ PASS [policies] renewal_consistency (within SLA)")# 3) Dictionary-based validations (ERROR severity)dim_channel      = spark.table(f"{DB_REF}.dim_channel")dim_product_line = spark.table(f"{DB_REF}.dim_product_line")dq_left_anti_ref(pol_silver, dim_channel, "Channel", "Channel",                 "channel_in_dictionary", "error", "policies", quarantine, "invalid_channel")dq_left_anti_ref(pol_silver, dim_product_line, "Product_Line", "Product_Line",                 "product_line_in_dictionary", "error", "policies", quarantine, "invalid_product_line")

%md## Cell 8 — Write Silver table & optimize

In [ ]:
# ---------- Cell 8 — Write Policies Silver (with MERGE support), Optimize & Vacuum ----------from pyspark.sql import functions as F# Preconditionsassert "pol_silver" in locals(), "pol_silver is missing — run prior cells."assert "DB_SILVER"  in locals(), "DB_SILVER is missing — run Cell 1."full_table = f"{DB_SILVER}.policies"# OPTIONAL: if you want to write to ADLS Gen2 silver container first, set PATH_POLICIES# Example:# STORAGE_ACCOUNT = "bupaclientstorage"# CONTAINER_SILVER = "silver"# PATH_POLICIES = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/policies"USE_EXTERNAL_PATH = "PATH_POLICIES" in locals() and PATH_POLICIES# Ensure DB existsspark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")# -------------------------# A) Write using write_silver helper (preferred: MERGE if you have a timestamp)# -------------------------# Decide write mode:# If you maintain an 'ingestion_ts' or 'last_modified_ts' column in pol_silver, we can MERGE idempotently.merge_ts_col = Nonefor candidate in ["last_modified_ts", "ingestion_ts", "updated_at", "processed_ts"]:    if candidate in pol_silver.columns:        merge_ts_col = candidate        breakif not USE_EXTERNAL_PATH:    try:        if merge_ts_col:            # Idempotent incremental write            write_silver(                pol_silver,                full_table,                key_cols=["Policy_ID"],                merge_ts_col=merge_ts_col,                mode="merge"            )        else:            # Batch overwrite (schema-evolving)            write_silver(                pol_silver,                full_table,                mode="overwrite"            )    except Exception as e:        # Safe inline fallback        print(f"[INFO] write_silver failed or unavailable → using inline writer. Details: {e}")        (pol_silver.write            .format("delta")            .mode("overwrite")            .option("overwriteSchema","true")            .saveAsTable(full_table))        print(f"[WRITE] Overwrote {full_table} with schema update.")# -------------------------# B) OPTIONAL: Write to external ADLS path and (re)register table on top# -------------------------else:    # Files first    (pol_silver.write        .format("delta")        .mode("overwrite")        .option("overwriteSchema","true")        .save(PATH_POLICIES))    print("✅ Wrote Policies to Delta path:", PATH_POLICIES)    # External table pointing to that path    spark.sql(f"DROP TABLE IF EXISTS {full_table}")    spark.sql(f"""    CREATE TABLE {full_table}    USING DELTA    LOCATION '{PATH_POLICIES}'    """)    print(f"✅ External table created: {full_table}")# -------------------------# OPTIMIZE / ZORDER (best effort)# -------------------------try:    spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Policy_ID, Policy_Start_Date)")    print("✅ OPTIMIZE complete.")except Exception as e:    print(f"[WARN] OPTIMIZE unavailable or skipped: {e}")# -------------------------# VACUUM (7 days retention) — maintenance# -------------------------try:    spark.sql(f"VACUUM {full_table} RETAIN 168 HOURS")    print("🧹 VACUUM complete (7 days).")except Exception as e:    print(f"[WARN] VACUUM unavailable or skipped: {e}")print("✅ Policies Silver written & maintained:", full_table)# Optional peek:# display(spark.table(full_table).limit(10))

In [ ]:
# ---------- Cell 8 — Write Policies Silver (with MERGE support), Optimize & Vacuum ----------from pyspark.sql import functions as F# Preconditionsassert "pol_silver" in locals(), "pol_silver is missing — run prior cells."assert "DB_SILVER"  in locals(), "DB_SILVER is missing — run Cell 1."full_table = f"{DB_SILVER}.policies"USE_EXTERNAL_PATH = "PATH_POLICIES" in locals() and PATH_POLICIES# Ensure DB existsspark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")# -------------------------# A) Write using write_silver helper (preferred: MERGE if you have a timestamp)# -------------------------merge_ts_col = Nonefor candidate in ["last_modified_ts", "ingestion_ts", "updated_at", "processed_ts"]:    if candidate in pol_silver.columns:        merge_ts_col = candidate        breakif not USE_EXTERNAL_PATH:    try:        if merge_ts_col:            write_silver(                pol_silver,                full_table,                key_cols=["Policy_ID"],                merge_ts_col=merge_ts_col,                mode="merge"            )        else:            write_silver(                pol_silver,                full_table,                mode="overwrite"            )    except Exception as e:        print(f"[INFO] write_silver failed or unavailable → using inline writer. Details: {e}")        (pol_silver.write            .format("delta")            .mode("overwrite")            .option("overwriteSchema","true")            .saveAsTable(full_table))        print(f"[WRITE] Overwrote {full_table} with schema update.")# -------------------------# B) OPTIONAL: Write to external ADLS path and (re)register table on top# -------------------------else:    (pol_silver.write        .format("delta")        .mode("overwrite")        .option("overwriteSchema","true")        .save(PATH_POLICIES))    print("✅ Wrote Policies to Delta path:", PATH_POLICIES)    spark.sql(f"DROP TABLE IF EXISTS {full_table}")    spark.sql(f"""    CREATE TABLE {full_table}    USING DELTA    LOCATION '{PATH_POLICIES}'    """)    print(f"✅ External table created: {full_table}")# -------------------------# C) Always write to external Delta path for backup or downstream use# -------------------------if "paths_silver" in locals() and "policies" in paths_silver:    (pol_silver.write        .format("delta")        .mode("overwrite")        .option("overwriteSchema", "true")        .save(paths_silver["policies"]))    print(f"✅ Wrote Policies to {paths_silver['policies']} Successfully:")# -------------------------# OPTIMIZE / ZORDER (best effort)# -------------------------try:    spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Policy_ID, Policy_Start_Date)")    print("✅ OPTIMIZE complete.")except Exception as e:    print(f"[WARN] OPTIMIZE unavailable or skipped: {e}")# -------------------------# VACUUM (7 days retention) — maintenance# -------------------------try:    spark.sql(f"VACUUM {full_table} RETAIN 168 HOURS")    print("🧹 VACUUM complete (7 days).")except Exception as e:    print(f"[WARN] VACUUM unavailable or skipped: {e}")print("✅ Policies Silver written & maintained:", full_table)# Optional peek:# display(spark.table(full_table).limit(10))

In [ ]:
"""# ============ Write Silver Policies & Optimize ============full_table = f"{DB_SILVER}.policies"(pol_silver.write  .format("delta")  .mode("overwrite")  .option("overwriteSchema", "true")  .saveAsTable(full_table))(pol_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths_silver["policies"]))print(f"✅ Wrote Policies to {paths_silver['policies']} Successfully:")optimize_zorder(full_table, ["Policy_ID","Policy_Start_Date"])print("✅ Registered Silver Policies in Metastore successfully.")display(spark.table(full_table).limit(10))"""

%md## Cell 9 — Run metrics (observability)

In [ ]:
# ============ Run Metrics ============def write_metric(name, value, context):    spark.createDataFrame([(name, str(value), context, datetime.utcnow())],                          "metric STRING, value STRING, context STRING, ts TIMESTAMP") \         .write.mode("append").saveAsTable(f"{DB_SILVER}.__run_metrics")write_metric("rowcount_policies_silver", pol_silver.count(), "policies_silver")write_metric("distinct_policy_ids", pol_silver.select("Policy_ID").distinct().count(), "policies_silver")display(spark.table(f"{DB_SILVER}.__run_metrics").orderBy(F.col("ts").desc()))

%md## Cell 10 — Exit cleanly (for job runs)

In [ ]:
dbutils.notebook.exit("SUCCESS: policies_silver completed")

%md# Policies after processing silver layer

In [ ]:
MAGIC %sqlselect * from bupa_silver.policies

%md# Differences b/w bronze policies and silver policies

In [ ]:
# Load DataFramesbronze_df = spark.table("bupa_bronze.policies")silver_df = spark.table("bupa_silver.policies")bronze_cols = set(bronze_df.columns)silver_cols = set(silver_df.columns)# Identify new, dropped, and common featuresnew_features = silver_cols - bronze_colsdropped_features = bronze_cols - silver_colscommon_features = sorted(bronze_cols & silver_cols)print(f"New features added in silver: {sorted(new_features)}")print(f"Features dropped in silver: {sorted(dropped_features)}")# Get data types as dictsbronze_types = dict(bronze_df.dtypes)silver_types = dict(silver_df.dtypes)# Align on Policy_ID for accurate comparisonbronze_aligned = bronze_df.select("Policy_ID", *common_features).dropDuplicates(["Policy_ID"])silver_aligned = silver_df.select("Policy_ID", *common_features).dropDuplicates(["Policy_ID"])joined = (    bronze_aligned.alias("bz")    .join(silver_aligned.alias("sv"), on="Policy_ID", how="outer"))# For each common feature, check for changes and count them, and compare typesfor col in common_features:    bronze_type = bronze_types.get(col)    silver_type = silver_types.get(col)    type_changed = bronze_type != silver_type    type_msg = (        f"Type changed: {bronze_type} → {silver_type}"        if type_changed else        f"Type unchanged: {bronze_type}"    )    n_diff = (        joined        .filter(            (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |            (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |            (joined[f"bz.{col}"] != joined[f"sv.{col}"])        )        .count()    )    if n_diff == 0 and not type_changed:        print(f"Feature '{col}' is unchanged between bronze and silver. {type_msg}")    else:        print(f"Feature '{col}' was transformed between bronze and silver in {n_diff} policies. {type_msg}")        # Show a sample of differences        sample_diff = (            joined            .filter(                (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |                (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |                (joined[f"bz.{col}"] != joined[f"sv.{col}"])            )            .select("Policy_ID", f"bz.{col}", f"sv.{col}")            .limit(5)        )        print(f"Sample differences for '{col}':")        display(sample_diff)# Explain new features and show sample datafor col in sorted(new_features):    print(f"Feature '{col}' added: Reason unknown, please check transformation logic.")    print(f"Sample data for new feature '{col}':")    display(silver_df.select("Policy_ID", col).limit(5))# Note dropped featuresfor col in sorted(dropped_features):    print(f"Feature '{col}' was present in bronze but dropped in silver.")

%md# A) Snapshot basics: row counts & distinct IDs

In [ ]:
from pyspark.sql import functions as FBRONZE_TBL = "bupa_bronze.policies"     # change if differentSILVER_TBL = "bupa_silver.policies"br = spark.table(BRONZE_TBL)sv = spark.table(SILVER_TBL)metrics = {    "rows_bronze": br.count(),    "rows_silver": sv.count(),    "distinct_policy_id_bronze": br.select("Policy_ID").distinct().count(),    "distinct_policy_id_silver": sv.select("Policy_ID").distinct().count(),}print(metrics)

%md# B) Data quality improvement deltas(Check null keys, bad dates, and invalid numeric values)

In [ ]:
def pct(x, n):     return 0.0 if n == 0 else round(100.0 * x / n, 4)# --- Bronze ---n_br = br.count()br_null_keys = br.filter(F.col("Policy_ID").isNull() | F.col("Customer_ID").isNull()).count()br_bad_dates = br.filter(    F.col("Policy_Start_Date").isNotNull() &    F.col("Policy_End_Date").isNotNull() &    (F.col("Policy_End_Date") < F.col("Policy_Start_Date"))).count()br_neg_money = br.filter(    (F.col("Sum_Insured_GBP") < 0) | (F.col("Annual_Premium_GBP") < 0)).count()br_discount_out = br.filter(    F.col("Discount_Offered_%").isNotNull() &     ((F.col("Discount_Offered_%") < 0) | (F.col("Discount_Offered_%") > 100))).count()# --- Silver ---n_sv = sv.count()sv_null_keys = sv.filter(F.col("Policy_ID").isNull() | F.col("Customer_ID").isNull()).count()sv_bad_dates = sv.filter(    F.col("Policy_Start_Date").isNotNull() &    F.col("Policy_End_Date").isNotNull() &    (F.col("Policy_End_Date") < F.col("Policy_Start_Date"))).count()sv_neg_money = sv.filter(    (F.col("Sum_Insured_GBP") < 0) | (F.col("Annual_Premium_GBP") < 0)).count()sv_discount_out = sv.filter(    F.col("Discount_Offered_%").isNotNull() &     ((F.col("Discount_Offered_%") < 0) | (F.col("Discount_Offered_%") > 100))).count()report = {    "bronze_null_keys_pct": pct(br_null_keys, n_br),    "silver_null_keys_pct": pct(sv_null_keys, n_sv),    "bronze_bad_date_order_pct": pct(br_bad_dates, n_br),    "silver_bad_date_order_pct": pct(sv_bad_dates, n_sv),    "bronze_negative_money_pct": pct(br_neg_money, n_br),    "silver_negative_money_pct": pct(sv_neg_money, n_sv),    "bronze_discount_out_pct": pct(br_discount_out, n_br),    "silver_discount_out_pct": pct(sv_discount_out, n_sv),}report

%md# C) FK progress vs Members (optional strict check)(Run only if Members Silver is ready; else skip this cell)

In [ ]:
def canon(col):    return F.upper(F.regexp_replace(F.trim(F.col(col)), r"[^A-Z0-9_]", ""))mem = spark.table("bupa_silver.members").select(canon("Policy_ID").alias("Policy_ID_canon")).dropDuplicates()# Bronze — directbr_fk_miss = (    br.withColumn("Policy_ID_canon", canon("Policy_ID"))      .join(mem, "Policy_ID_canon", "left_anti")).count()br_fk_pct = pct(br_fk_miss, br.count())# Silver — should be ~0 after canonical + enforcementsv_fk_miss = (    sv.withColumn("Policy_ID_canon", canon("Policy_ID"))      .join(mem, "Policy_ID_canon", "left_anti")).count()sv_fk_pct = pct(sv_fk_miss, sv.count())print({"bronze_fk_missing_pct": br_fk_pct, "silver_fk_missing_pct": sv_fk_pct})

%md# D) Feature coverage check (Silver-only)

In [ ]:
exists = {c: (c in sv.columns) for c in [    "Policy_Duration_Days",    "Renewal_Offered_Flag",    "Renewal_Accepted_Flag",    "Discount_Offered_%",    "Channel",    "record_hash",    "created_at",    "source_system"]}exists